# Pipeline ETL — Análise da Alfabetização no Brasil
#### *Ingestão (BigQuery / Base dos Dados), Camadas Bronze/Silver/Gold e Qualidade de Dados*
### Python + Pandas

---

**Objetivo:** Construir um pipeline ETL completo para o **Indicador Criança Alfabetizada**, desde a ingestão dos dados públicos do INEP (via **BigQuery / Base dos Dados**) até a entrega de dados prontos para análise, aplicando boas práticas de qualidade de dados e arquitetura em camadas (*Medallion Architecture*).

**Contexto:** O **Compromisso Nacional Criança Alfabetizada** estabelece que todas as crianças estejam alfabetizadas até o final do 2º ano do Ensino Fundamental até **2030**. A partir da *Pesquisa Alfabetiza Brasil* (2023), definiu-se o ponto de corte de **743 pontos** na escala Saeb, a partir do qual a criança é considerada alfabetizada.

---

### Estrutura da Demo

| Etapa | Descrição |
|---|---|
| **1. Setup** | Ambiente, autenticação GCP (BigQuery + GCS) e caminhos do Data Lake |
| **2. Ingestão (Bronze)** | Leitura das tabelas do INEP via BigQuery + armazenamento *raw* em Parquet |
| **3. Limpeza (Silver)** | Tratamento, padronização de tipos, normalização de chaves e deduplicação |
| **4. Enriquecimento (Gold)** | Integração das bases, comparação metas × resultados e métricas de consumo |
| **5. Qualidade de Dados** | Validações, regras e relatório de qualidade por camada |

### Fontes — dataset `br_inep_avaliacao_alfabetizacao` (Base dos Dados)

| Entidade | Descrição |
|---|---|
| **UF** | Indicadores de alfabetização agregados por Unidade Federativa |
| **Município** | Indicadores de alfabetização agregados por município |
| **Meta Alfabetização — Brasil** | Metas nacionais 2024–2030 |
| **Meta Alfabetização — UF** | Metas por estado |
| **Meta Alfabetização — Município** | Metas por município |
| **Alunos** | Indicadores de desempenho dos estudantes (microdados agregados) |

---

## IMPORTANTE:
Reprocessar do zero (baixar do BigQuery): rode Célula 4, 6 e 9.  
Retomar / só inspecionar (sem BigQuery): rode Célula 4, 8 e 9.

---
## ⚙️ Etapa 1 — Setup do Ambiente

Configuração das dependências de apoio, autenticação no **Google Cloud (BigQuery + GCS)** e criação da estrutura local do **Data Lake** (`bronze` / `silver` / `gold` / `quality_reports`).

In [ ]:
# %pip install colorama tabulate --quiet

Note: you may need to restart the kernel to use updated packages.


In [ ]:
# ============================================================
# IMPORTS E CONFIGURAÇÕES GLOBAIS
# ============================================================
import os
import json
import hashlib
import warnings
from pathlib import Path
from datetime import datetime, timezone

import pandas as pd
import numpy as np
import basedosdados as bd
from dotenv import load_dotenv, find_dotenv
from colorama import Fore, Style, init
from tabulate import tabulate

warnings.filterwarnings('ignore')
init(autoreset=True)

# ─── 1. Autenticação GCP ───
# O .env contém: GOOGLE_APPLICATION_CREDENTIALS=./google-credentials.json
dotenv_path = find_dotenv(usecwd=True)
load_dotenv(dotenv_path)

# Raiz do projeto
# Serve de âncora para os caminhos (independe de onde o notebook roda).
PROJECT_ROOT = Path(dotenv_path).parent if dotenv_path else Path.cwd()

# Resolve o caminho das credenciais
_cred_raw = os.getenv("GOOGLE_APPLICATION_CREDENTIALS", "./google-credentials.json")
CRED_PATH = (PROJECT_ROOT / _cred_raw).resolve()
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = str(CRED_PATH)

# ─── 2. Configuração do projeto GCP / BigQuery ───
# billing_project_id é OBRIGATÓRIO no basedosdados — passamos direto no Python.
GCP_PROJECT_ID = os.getenv("GCP_PROJECT_ID", "postech-alfabetizacao")
GCS_BUCKET     = os.getenv("GCS_BUCKET", "postech-alfabetizacao-data-lake")  # uso opcional (upload p/ GCS)

# ─── 3. Estrutura local do Data Lake (Medallion) ───
BASE_PATH    = PROJECT_ROOT / "data_lake"
BRONZE_PATH  = BASE_PATH / "bronze"
SILVER_PATH  = BASE_PATH / "silver"
GOLD_PATH    = BASE_PATH / "gold"
QUALITY_PATH = BASE_PATH / "quality_reports"

for path in [BRONZE_PATH, SILVER_PATH, GOLD_PATH, QUALITY_PATH]:
    path.mkdir(parents=True, exist_ok=True)

# ─── 4. Metadados de execução ───
INGESTION_TIMESTAMP = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")

# ─── Validações rápidas de ambiente ───
print(f"{Fore.GREEN}✅ Ambiente configurado!\n")
print(f"{Fore.CYAN}🔑 Credenciais GCP : {CRED_PATH}")
print(f"   Arquivo existe?  : {'✅ SIM' if CRED_PATH.exists() else '❌ NÃO ENCONTRADO'}")
print(f"{Fore.CYAN}☁️  Projeto GCP     : {GCP_PROJECT_ID}")
print(f"   Bucket GCS (opc.): gs://{GCS_BUCKET}")
print(f"\n{Fore.CYAN}🗂️  Data Lake local : {BASE_PATH}")
print(f"   ├── bronze/           → Dados brutos (raw)")
print(f"   ├── silver/           → Dados limpos e normalizados")
print(f"   ├── gold/             → Dados agregados para consumo")
print(f"   └── quality_reports/  → Relatórios de qualidade")
print(f"\n{Fore.YELLOW}🕒 Timestamp de ingestão: {INGESTION_TIMESTAMP}")

✅ Ambiente configurado!

🔑 Credenciais GCP : D:\diego\01_projects\postech-challenge-2\google-credentials.json
   Arquivo existe?  : ✅ SIM
☁️  Projeto GCP     : postech-alfabetizacao
   Bucket GCS (opc.): gs://postech-alfabetizacao-data-lake

🗂️  Data Lake local : d:\diego\01_projects\postech-challenge-2\data_lake
   ├── bronze/           → Dados brutos (raw)
   ├── silver/           → Dados limpos e normalizados
   ├── gold/             → Dados agregados para consumo
   └── quality_reports/  → Relatórios de qualidade

🕒 Timestamp de ingestão: 20260630_011028


---
## 🥉 Etapa 2 — Camada BRONZE: Ingestão de Dados

A **camada Bronze** armazena os dados **exatamente como vieram da fonte** — sem transformações. Isso garante **rastreabilidade** e a possibilidade de **reprocessamento** a qualquer momento.

**Fonte utilizada:** dataset público [`br_inep_avaliacao_alfabetizacao`](https://basedosdados.org/dataset/073a39d4-89cf-4068-b1e8-34ed0d9c0b72) na **Base dos Dados**, lido via **BigQuery** com o SDK `basedosdados`.

**Entidades ingeridas (6):**

| Entidade | Tabela no BigQuery | Papel |
|---|---|---|
| `uf` | `uf` | Indicador agregado por UF |
| `municipio` | `municipio` | Indicador agregado por município |
| `alunos` | `alunos` | Microdado em nível de aluno (~3,9M linhas) |
| `meta_brasil` | `meta_alfabetizacao_brasil` | Metas nacionais 2024–2030 |
| `meta_uf` | `meta_alfabetizacao_uf` | Metas por UF |
| `meta_municipio` | `meta_alfabetizacao_municipio` | Metas por município |

  
> 💡 **Boa prática:** sempre adicionar **metadados de ingestão** aos dados brutos. Aqui registramos `_ingestion_timestamp`, `_source_system`, `_source_dataset`, `_source_table` e um `_record_hash` por registro (linhagem e detecção de mudanças).

> 💰 **FinOps:** o scan no BigQuery é faturado no *seu* projeto (`billing_project_id`), enquanto os dados vêm do projeto público `basedosdados`. As tabelas do INEP são pequenas o suficiente para caberem no nível gratuito (1 TB/mês).

In [2]:
# ============================================================
# ETAPA 2.1 — FUNÇÃO DE INGESTÃO GENÉRICA (BigQuery / Base dos Dados)
# ============================================================
# Substitui a ingestão via HTTP do template original.
# Lê uma tabela pública do BigQuery (Base dos Dados), adiciona metadados
# de linhagem e salva os dados brutos na camada Bronze.
#
# 💰 FinOps: bd.read_table() escaneia a tabela inteira no BigQuery. As tabelas
#    do INEP são pequenas (dezenas a ~24k linhas) => custo desprezível, bem
#    dentro do 1 TB/mês gratuito. O scan é faturado no SEU projeto
#    (billing_project_id); os dados vêm do projeto público 'basedosdados'.

def ingest_from_bigquery(
    dataset_id: str,
    table_id: str,
    entity_name: str,
    limit: int = None,
) -> pd.DataFrame:
    """
    Ingere uma tabela do BigQuery (Base dos Dados) e salva na camada Bronze.

    Args:
        dataset_id:  ID do dataset (ex: 'br_inep_avaliacao_alfabetizacao')
        table_id:    ID da tabela  (ex: 'uf', 'municipio', 'meta_alfabetizacao_brasil')
        entity_name: Nome amigável usado no arquivo de saída (ex: 'uf', 'meta_brasil')
        limit:       (opcional) limita nº de linhas — útil p/ testes. None = tabela completa.

    Returns:
        DataFrame com os dados brutos + metadados de linhagem.
    """
    source_ref = f"basedosdados.{dataset_id}.{table_id}"
    print(f"{Fore.CYAN}📥 Iniciando ingestão de [{entity_name}]...")
    print(f"   Fonte BigQuery: {source_ref}" + (f"  (LIMIT {limit})" if limit else ""))

    try:
        # ── Leitura via Base dos Dados (BigQuery) ──
        df = bd.read_table(
            dataset_id=dataset_id,
            table_id=table_id,
            billing_project_id=GCP_PROJECT_ID,
            limit=limit,
        )

        # ── Metadados de linhagem (rastreabilidade / reprocessamento) ──
        df['_ingestion_timestamp'] = INGESTION_TIMESTAMP
        df['_source_system']       = 'base_dos_dados_bigquery'
        df['_source_dataset']      = dataset_id
        df['_source_table']        = table_id
        df['_record_hash'] = df.drop(
            columns=['_ingestion_timestamp', '_source_system',
                     '_source_dataset', '_source_table'],
            errors='ignore'
        ).apply(lambda row: hashlib.md5(str(row.values).encode()).hexdigest(), axis=1)

        # ── Persistência na camada Bronze (Parquet) ──
        output_file = BRONZE_PATH / f"{entity_name}_raw_{INGESTION_TIMESTAMP}.parquet"
        df.to_parquet(output_file, index=False)

        print(f"{Fore.GREEN}   ✅ {len(df):,} registros ingeridos")
        print(f"   💾 Salvo em: {output_file.relative_to(PROJECT_ROOT)}")
        print(f"   📋 Colunas ({len(df.columns)}): {list(df.columns)}")

        return df

    except Exception as e:
        print(f"{Fore.RED}   ❌ ERRO ao ingerir [{entity_name}]: {type(e).__name__}")
        print(f"{Fore.RED}      {e}")
        print(f"{Fore.YELLOW}      Verifique: billing_project_id, permissões da Service Account "
              f"(BigQuery) e se dataset_id/table_id existem.")
        raise


print(f"{Fore.GREEN}✅ Função de ingestão (BigQuery) definida!")

✅ Função de ingestão (BigQuery) definida!


In [ ]:
# ============================================================
# ETAPA 2.2 — INGESTÃO MULTI-ENTIDADE (6 tabelas do INEP via BigQuery)
# ============================================================
# ID completo da tabela: basedosdados.br_inep_avaliacao_alfabetizacao.alunos

print(f"{Fore.YELLOW}{'='*60}")
print(f"{Fore.YELLOW}🚀 INICIANDO PIPELINE DE INGESTÃO (BigQuery → Bronze)")
print(f"{Fore.YELLOW}{'='*60}\n")

DATASET_ID = "br_inep_avaliacao_alfabetizacao"

# Mapeamento: nome usado no arquivo -> table_id real no BigQuery
entities = {
    # ── Tabelas do indicador (desempenho) ──
    'uf':             'uf',
    'municipio':      'municipio',
    'alunos':         'alunos',                       # schema/tamanho a confirmar na inspeção
    # ── Tabelas de metas ──
    'meta_brasil':    'meta_alfabetizacao_brasil',
    'meta_uf':        'meta_alfabetizacao_uf',
    'meta_municipio': 'meta_alfabetizacao_municipio',
}

bronze_dfs = {}
for entity_name, table_id in entities.items():
    bronze_dfs[entity_name] = ingest_from_bigquery(DATASET_ID, table_id, entity_name)
    print()

# ── Resumo da ingestão ──
print(f"{Fore.GREEN}{'='*60}")
print(f"{Fore.GREEN}✅ INGESTÃO CONCLUÍDA")
print(f"{Fore.GREEN}   Entidades ingeridas: {len(bronze_dfs)}")
print(f"{Fore.GREEN}   Total de registros : {sum(len(df) for df in bronze_dfs.values()):,}")
print(f"{Fore.GREEN}{'='*60}\n")

resumo = [(e, f"{len(df):,}", df.shape[1]) for e, df in bronze_dfs.items()]
print(tabulate(resumo, headers=['Entidade', 'Linhas', 'Colunas'], tablefmt='rounded_outline'))

🚀 INICIANDO PIPELINE DE INGESTÃO (BigQuery → Bronze)

📥 Iniciando ingestão de [uf]...
   Fonte BigQuery: basedosdados.br_inep_avaliacao_alfabetizacao.uf
Please visit this URL to authorize this application: https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=262006177488-3425ks60hkk80fssi9vpohv88g6q1iqd.apps.googleusercontent.com&redirect_uri=http%3A%2F%2Flocalhost%3A8080%2F&scope=https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fcloud-platform&state=78OSU7QqEjKNwdoAftDJ8yhVnSxRN6&code_challenge=OQjWfsYPvw4OoI_MPnudTSEoKSjwzP3jloTeuCut_QE&code_challenge_method=S256&prompt=consent&access_type=offline
Downloading: 100%|          |██████████|
   ✅ 145 registros ingeridos
   💾 Salvo em: data_lake\bronze\uf_raw_20260630_011028.parquet
   📋 Colunas (20): ['ano', 'sigla_uf', 'serie', 'rede', 'taxa_alfabetizacao', 'media_portugues', 'proporcao_aluno_nivel_0', 'proporcao_aluno_nivel_1', 'proporcao_aluno_nivel_2', 'proporcao_aluno_nivel_3', 'proporcao_aluno_nivel_4', 'proporcao_aluno_n

In [5]:
# ============================================================
# ETAPA 2.3 — RECARREGAR A CAMADA BRONZE A PARTIR DO DISCO
# ============================================================
# Reconstrói `bronze_dfs` lendo os Parquet já gravados, SEM re-baixar do
# BigQuery. Use isto para:
#   • retomar o trabalho noutro dia (sem re-ingerir 'alunos' = 3,87M linhas);
#   • inspecionar/continuar a partir do disco;
#   • quem clonou o repo e já tem os Parquet da Bronze.
# Pega sempre a ingestão MAIS RECENTE de cada entidade (se houver vários
# timestamps). A lista ENTITIES é redefinida aqui de propósito, para esta
# célula funcionar como ponto de entrada independente (sem depender da Célula 6).

import glob

ENTITIES = ['uf', 'municipio', 'alunos', 'meta_brasil', 'meta_uf', 'meta_municipio']

bronze_dfs = {}
print(f"{Fore.YELLOW}📂 Recarregando camada Bronze de: {BRONZE_PATH}\n")

for entity in ENTITIES:
    matches = sorted(glob.glob(str(BRONZE_PATH / f"{entity}_raw_*.parquet")))
    if not matches:
        print(f"{Fore.RED}   ⚠️  Nenhum Parquet para [{entity}] "
              f"— rode a Célula 6 (ingestão) primeiro.")
        continue
    latest = matches[-1]                       # nome ordena por timestamp -> mais recente
    bronze_dfs[entity] = pd.read_parquet(latest)
    print(f"{Fore.GREEN}   ✅ {entity:14s} ← {os.path.basename(latest)} "
          f"({len(bronze_dfs[entity]):,} linhas)")

print(f"\n{Fore.CYAN}bronze_dfs reconstruído: {len(bronze_dfs)} entidades, "
      f"{sum(len(d) for d in bronze_dfs.values()):,} registros (sem tocar no BigQuery).")

📂 Recarregando camada Bronze de: d:\diego\01_projects\postech-challenge-2\data_lake\bronze

   ✅ uf             ← uf_raw_20260630_011028.parquet (145 linhas)
   ✅ municipio      ← municipio_raw_20260630_011028.parquet (23,995 linhas)
   ✅ alunos         ← alunos_raw_20260630_011028.parquet (3,867,999 linhas)
   ✅ meta_brasil    ← meta_brasil_raw_20260630_011028.parquet (3 linhas)
   ✅ meta_uf        ← meta_uf_raw_20260630_011028.parquet (81 linhas)
   ✅ meta_municipio ← meta_municipio_raw_20260630_011028.parquet (10,704 linhas)

bronze_dfs reconstruído: 6 entidades, 3,902,927 registros (sem tocar no BigQuery).


In [4]:
# ============================================================
# ETAPA 2.3 — INSPEÇÃO DOS DADOS BRUTOS
# ============================================================
# Sempre inspecionar os dados antes de processá-los!

print(f"{Fore.YELLOW}🔎 PRÉVIA DOS DADOS BRUTOS (Bronze)\n")

for entity, df in bronze_dfs.items():
    print(f"{Fore.CYAN}{'─'*60}")
    print(f"{Fore.CYAN}  Entidade: {entity.upper()}")
    print(f"   Shape: {df.shape}  |  Nulos totais: {df.isnull().sum().sum():,}")
    print(f"   Tipos: { {col: str(dtype) for col, dtype in df.dtypes.items()} }")
    print()
    display(df.head(3))
    print()

🔎 PRÉVIA DOS DADOS BRUTOS (Bronze)

────────────────────────────────────────────────────────────
  Entidade: UF
   Shape: (145, 20)  |  Nulos totais: 630
   Tipos: {'ano': 'Int64', 'sigla_uf': 'object', 'serie': 'object', 'rede': 'object', 'taxa_alfabetizacao': 'float64', 'media_portugues': 'float64', 'proporcao_aluno_nivel_0': 'float64', 'proporcao_aluno_nivel_1': 'float64', 'proporcao_aluno_nivel_2': 'float64', 'proporcao_aluno_nivel_3': 'float64', 'proporcao_aluno_nivel_4': 'float64', 'proporcao_aluno_nivel_5': 'float64', 'proporcao_aluno_nivel_6': 'float64', 'proporcao_aluno_nivel_7': 'float64', 'proporcao_aluno_nivel_8': 'float64', '_ingestion_timestamp': 'object', '_source_system': 'object', '_source_dataset': 'object', '_source_table': 'object', '_record_hash': 'object'}



,ano,sigla_uf,serie,rede,taxa_alfabetizacao,media_portugues,proporcao_aluno_nivel_0,proporcao_aluno_nivel_1,proporcao_aluno_nivel_2,proporcao_aluno_nivel_3,proporcao_aluno_nivel_4,proporcao_aluno_nivel_5,proporcao_aluno_nivel_6,proporcao_aluno_nivel_7,proporcao_aluno_nivel_8,_ingestion_timestamp,_source_system,_source_dataset,_source_table,_record_hash
0,2023,AM,2,3,49.20,733.6637,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,20260630_011028,base_dos_dados_bigquery,br_inep_avaliacao_alfabetizacao,uf,42ee6a943b5c1b9714c8e978c57f248b
1,2023,PB,2,2,55.23,744.8152,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,20260630_011028,base_dos_dados_bigquery,br_inep_avaliacao_alfabetizacao,uf,eca51cb864fb5c8429b4b562aa79eee6
2,2023,PR,2,5,73.12,757.2146,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,20260630_011028,base_dos_dados_bigquery,br_inep_avaliacao_alfabetizacao,uf,586978b39f93e4d089e198ec3945236a



────────────────────────────────────────────────────────────
  Entidade: MUNICIPIO
   Shape: (23995, 20)  |  Nulos totais: 103,923
   Tipos: {'ano': 'Int64', 'id_municipio': 'object', 'serie': 'object', 'rede': 'object', 'taxa_alfabetizacao': 'float64', 'media_portugues': 'float64', 'proporcao_aluno_nivel_0': 'float64', 'proporcao_aluno_nivel_1': 'float64', 'proporcao_aluno_nivel_2': 'float64', 'proporcao_aluno_nivel_3': 'float64', 'proporcao_aluno_nivel_4': 'float64', 'proporcao_aluno_nivel_5': 'float64', 'proporcao_aluno_nivel_6': 'float64', 'proporcao_aluno_nivel_7': 'float64', 'proporcao_aluno_nivel_8': 'float64', '_ingestion_timestamp': 'object', '_source_system': 'object', '_source_dataset': 'object', '_source_table': 'object', '_record_hash': 'object'}



,ano,id_municipio,serie,rede,taxa_alfabetizacao,media_portugues,proporcao_aluno_nivel_0,proporcao_aluno_nivel_1,proporcao_aluno_nivel_2,proporcao_aluno_nivel_3,proporcao_aluno_nivel_4,proporcao_aluno_nivel_5,proporcao_aluno_nivel_6,proporcao_aluno_nivel_7,proporcao_aluno_nivel_8,_ingestion_timestamp,_source_system,_source_dataset,_source_table,_record_hash
0,2023,1100031,2,3,69.10,767.8763,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,20260630_011028,base_dos_dados_bigquery,br_inep_avaliacao_alfabetizacao,municipio,d1b6df4e80959ce1a5082a62c2f1db0c
1,2023,1100072,2,3,58.20,747.8918,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,20260630_011028,base_dos_dados_bigquery,br_inep_avaliacao_alfabetizacao,municipio,4cda7a8d76f3afa7a02c2237a0868043
2,2023,1100189,2,5,69.73,762.4062,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,20260630_011028,base_dos_dados_bigquery,br_inep_avaliacao_alfabetizacao,municipio,5b53ac3da3f4a8d4c8a16760b6a46b74



────────────────────────────────────────────────────────────
  Entidade: ALUNOS
   Shape: (3867999, 17)  |  Nulos totais: 1,026,676
   Tipos: {'ano': 'Int64', 'id_municipio': 'object', 'id_escola': 'object', 'id_aluno': 'object', 'caderno': 'object', 'serie': 'object', 'rede': 'object', 'presenca': 'object', 'preenchimento_caderno': 'object', 'alfabetizado': 'object', 'proficiencia': 'float64', 'peso_aluno': 'float64', '_ingestion_timestamp': 'object', '_source_system': 'object', '_source_dataset': 'object', '_source_table': 'object', '_record_hash': 'object'}



...

,ano,rede,taxa_alfabetizacao,meta_alfabetizacao_2024,meta_alfabetizacao_2025,meta_alfabetizacao_2026,meta_alfabetizacao_2027,meta_alfabetizacao_2028,meta_alfabetizacao_2029,meta_alfabetizacao_2030,percentual_participacao,_ingestion_timestamp,_source_system,_source_dataset,_source_table,_record_hash
0,2025,Pública,66.0,60.0,64.00,67.00,71.00,74.00,77.00,80.0,88.00,20260630_011028,base_dos_dados_bigquery,br_inep_avaliacao_alfabetizacao,meta_alfabetizacao_brasil,fc91a9077697cf8711b674e9889d2cab
1,2024,Pública,59.2,59.9,63.77,67.47,70.97,74.23,77.24,80.0,87.37,20260630_011028,base_dos_dados_bigquery,br_inep_avaliacao_alfabetizacao,meta_alfabetizacao_brasil,14eac9e1c9a7e2bd9221c6cdd7647110
2,2023,Pública,55.9,59.9,63.77,67.47,70.97,74.23,77.24,80.0,86.00,20260630_011028,base_dos_dados_bigquery,br_inep_avaliacao_alfabetizacao,meta_alfabetizacao_brasil,5730997b67cf1c69fa2b61018ffe34c0



────────────────────────────────────────────────────────────
  Entidade: META_UF
   Shape: (81, 17)  |  Nulos totais: 30
   Tipos: {'ano': 'Int64', 'sigla_uf': 'object', 'rede': 'object', 'taxa_alfabetizacao': 'float64', 'meta_alfabetizacao_2024': 'float64', 'meta_alfabetizacao_2025': 'float64', 'meta_alfabetizacao_2026': 'float64', 'meta_alfabetizacao_2027': 'float64', 'meta_alfabetizacao_2028': 'float64', 'meta_alfabetizacao_2029': 'float64', 'meta_alfabetizacao_2030': 'float64', 'percentual_participacao': 'float64', '_ingestion_timestamp': 'object', '_source_system': 'object', '_source_dataset': 'object', '_source_table': 'object', '_record_hash': 'object'}



,ano,sigla_uf,rede,taxa_alfabetizacao,meta_alfabetizacao_2024,meta_alfabetizacao_2025,meta_alfabetizacao_2026,meta_alfabetizacao_2027,meta_alfabetizacao_2028,meta_alfabetizacao_2029,meta_alfabetizacao_2030,percentual_participacao,_ingestion_timestamp,_source_system,_source_dataset,_source_table,_record_hash
0,2024,RR,Pública,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,20260630_011028,base_dos_dados_bigquery,br_inep_avaliacao_alfabetizacao,meta_alfabetizacao_uf,ed1075837eda038b42ab9e284eeb4fb2
1,2023,RR,Pública,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,20260630_011028,base_dos_dados_bigquery,br_inep_avaliacao_alfabetizacao,meta_alfabetizacao_uf,375606ac4631d6049145aa9f6095f06e
2,2024,SE,Pública,38.39,38.3,45.9,53.6,61.2,68.3,74.6,80.0,92.84,20260630_011028,base_dos_dados_bigquery,br_inep_avaliacao_alfabetizacao,meta_alfabetizacao_uf,69f2f1143b6b896ef6e7f194280ebb72



────────────────────────────────────────────────────────────
  Entidade: META_MUNICIPIO
   Shape: (10704, 18)  |  Nulos totais: 600
   Tipos: {'ano': 'Int64', 'id_municipio': 'object', 'rede': 'object', 'taxa_alfabetizacao': 'float64', 'meta_alfabetizacao_2024': 'float64', 'meta_alfabetizacao_2025': 'float64', 'meta_alfabetizacao_2026': 'float64', 'meta_alfabetizacao_2027': 'float64', 'meta_alfabetizacao_2028': 'float64', 'meta_alfabetizacao_2029': 'float64', 'meta_alfabetizacao_2030': 'float64', 'nivel_alfabetizacao': 'Int64', 'percentual_participacao': 'float64', '_ingestion_timestamp': 'object', '_source_system': 'object', '_source_dataset': 'object', '_source_table': 'object', '_record_hash': 'object'}



,ano,id_municipio,rede,taxa_alfabetizacao,meta_alfabetizacao_2024,meta_alfabetizacao_2025,meta_alfabetizacao_2026,meta_alfabetizacao_2027,meta_alfabetizacao_2028,meta_alfabetizacao_2029,meta_alfabetizacao_2030,nivel_alfabetizacao,percentual_participacao,_ingestion_timestamp,_source_system,_source_dataset,_source_table,_record_hash
0,2023,4301750,Municipal,NaN,NaN,14.05,23.65,37.0,52.68,67.85,80.0,<NA>,NaN,20260630_011028,base_dos_dados_bigquery,br_inep_avaliacao_alfabetizacao,meta_alfabetizacao_municipio,d599e2a6152fe5e49952a7f9c07e7e02
1,2024,4301750,Municipal,4.40,NaN,14.05,23.65,37.0,52.68,67.85,80.0,0,92.59,20260630_011028,base_dos_dados_bigquery,br_inep_avaliacao_alfabetizacao,meta_alfabetizacao_municipio,f8c0b9945b3bb6fe09ea207f9311f87e
2,2024,2406908,Municipal,42.86,7.94,14.05,23.65,37.0,52.68,67.85,80.0,1,84.00,20260630_011028,base_dos_dados_bigquery,br_inep_avaliacao_alfabetizacao,meta_alfabetizacao_municipio,dd484f706727c46b70499c6b432f3f2d
